In [1]:
!pip install -q sentence-transformers chromadb tqdm torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.1 MB/s e

In [2]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

GPU Available: True
GPU Name: Tesla T4


In [7]:
"""
Embed optimized CUBE chunks into ChromaDB with rich metadata filtering
Optimized for Google Colab T4 GPU with BGE-M3 model
"""

import json
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from typing import List, Dict
from tqdm import tqdm
import os
import torch


class CUBEChunkEmbedder:
    def __init__(
        self,
        chunks_file: str,
        collection_name: str = "cube_docs_optimized",
        db_path: str = "/content/chroma_db",
        model_name: str = "BAAI/bge-m3",
        use_gpu: bool = True
    ):
        self.chunks_file = chunks_file
        self.collection_name = collection_name
        self.db_path = db_path
        self.model_name = model_name

        # GPU configuration
        self.device = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"

        # Initialize
        self.chunks_data = None
        self.model = None
        self.client = None
        self.collection = None

    def load_chunks(self):
        """Load chunks from JSON file"""
        print(f"📂 Loading chunks from {self.chunks_file}...")
        with open(self.chunks_file, 'r', encoding='utf-8') as f:
            self.chunks_data = json.load(f)

        total_chunks = len(self.chunks_data['chunks'])
        print(f"✓ Loaded {total_chunks} chunks")

        # Print statistics
        stats = self.chunks_data.get('statistics', {})
        print(f"  - Page chunks: {stats.get('page_chunks', 0)}")
        print(f"  - Synthetic chunks: {stats.get('synthetic_chunks', 0)}")
        print(f"  - Avg tokens: {stats.get('avg_tokens', 0):.0f}")

    def initialize_model(self):
        """Initialize embedding model with GPU support"""
        print(f"\n🤖 Loading embedding model: {self.model_name}...")
        print(f"   Device: {self.device.upper()}")

        if self.device == "cuda":
            print(f"   GPU: {torch.cuda.get_device_name(0)}")
            print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

        self.model = SentenceTransformer(self.model_name, device=self.device)
        print("✓ Model loaded successfully")

    def initialize_chromadb(self):
        """Initialize ChromaDB client and collection"""
        print(f"\n🗄️  Initializing ChromaDB at {self.db_path}...")

        # Create directory if it doesn't exist
        os.makedirs(self.db_path, exist_ok=True)

        # Initialize ChromaDB client
        self.client = chromadb.PersistentClient(
            path=self.db_path,
            settings=Settings(
                anonymized_telemetry=False,
                allow_reset=True
            )
        )

        # Delete existing collection if it exists
        try:
            self.client.delete_collection(name=self.collection_name)
            print(f"  ⚠️  Deleted existing collection: {self.collection_name}")
        except:
            pass

        # Create new collection
        self.collection = self.client.create_collection(
            name=self.collection_name,
            metadata={"description": "CUBE Banking Documentation - BGE-M3 Embeddings"}
        )
        print(f"✓ Created collection: {self.collection_name}")

    def prepare_metadata_for_chroma(self, metadata: Dict) -> Dict:
        """Prepare metadata for ChromaDB (convert lists to strings)"""
        chroma_metadata = {}

        for key, value in metadata.items():
            if value is None:
                continue
            elif isinstance(value, (list, set)):
                # Convert lists/sets to comma-separated strings
                chroma_metadata[key] = ",".join(str(v) for v in value)
            elif isinstance(value, dict):
                # Skip complex dicts or stringify them
                chroma_metadata[key] = json.dumps(value)
            elif isinstance(value, bool):
                chroma_metadata[key] = str(value)
            elif isinstance(value, (int, float)):
                chroma_metadata[key] = value
            else:
                chroma_metadata[key] = str(value)

        return chroma_metadata

    def embed_chunks(self, batch_size: int = 16):
        """Embed all chunks and add to ChromaDB"""
        chunks = self.chunks_data['chunks']
        print(f"\n🔮 Embedding {len(chunks)} chunks...")

        # Adjust batch size for GPU
        if self.device == "cuda":
            batch_size = 16  # Optimal for T4 GPU with BGE-M3
            print(f"   Using GPU-optimized batch size: {batch_size}")
        else:
            batch_size = 8  # Conservative for CPU
            print(f"   Using CPU batch size: {batch_size}")

        # Prepare data
        documents = []
        metadatas = []
        ids = []

        for idx, chunk in enumerate(tqdm(chunks, desc="Preparing chunks")):
            chunk_id = chunk['metadata'].get('chunk_id', f'chunk_{idx}')
            content = chunk['content']
            metadata = self.prepare_metadata_for_chroma(chunk['metadata'])

            # Add token count to metadata
            metadata['token_count'] = chunk['tokens']

            documents.append(content)
            metadatas.append(metadata)
            ids.append(chunk_id)

        # Generate embeddings in batches
        print(f"Generating embeddings with BGE-M3...")
        all_embeddings = []

        for i in tqdm(range(0, len(documents), batch_size), desc="Embedding batches"):
            batch = documents[i:i + batch_size]

            # BGE-M3 encoding with normalization (best for retrieval)
            embeddings = self.model.encode(
                batch,
                batch_size=batch_size,
                show_progress_bar=False,
                normalize_embeddings=True,  # Important for cosine similarity
                convert_to_tensor=False
            )
            all_embeddings.extend(embeddings.tolist())

        # Add to ChromaDB
        print("Adding to ChromaDB...")
        self.collection.add(
            embeddings=all_embeddings,
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )

        print(f"✓ Successfully embedded and stored {len(chunks)} chunks")

    def verify_collection(self):
        """Verify the collection was created correctly"""
        print("\n🔍 Verifying collection...")

        count = self.collection.count()
        print(f"  Total documents in collection: {count}")

        # Get sample
        sample = self.collection.peek(limit=1)
        if sample['documents']:
            print(f"  Sample document length: {len(sample['documents'][0])} chars")
            print(f"  Sample metadata keys: {list(sample['metadatas'][0].keys())}")

    def test_queries(self):
        """Test some sample queries"""
        print("\n🧪 Testing sample queries...")

        test_queries = [
            "How to open an NRI account?",
            "What is the NPC clearance process?",
            "Risk classification requirements",
            "Admin API sequence for account creation"
        ]

        for query in test_queries:
            print(f"\n  Query: '{query}'")

            # Generate query embedding with BGE-M3
            query_embedding = self.model.encode(
                [query],
                normalize_embeddings=True,
                convert_to_tensor=False
            )[0].tolist()

            # Search
            results = self.collection.query(
                query_embeddings=[query_embedding],
                n_results=3
            )

            print(f"  Top results:")
            for i, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0]), 1):
                page_name = metadata.get('page_name', metadata.get('title', 'N/A'))
                book_name = metadata.get('book_name', 'Synthetic')
                print(f"    {i}. {page_name} (from: {book_name})")
                print(f"       Preview: {doc[:100]}...")

    def run(self):
        """Run the complete embedding pipeline"""
        print("\n" + "="*70)
        print("CUBE DOCUMENTATION EMBEDDING - BGE-M3 on GPU")
        print("="*70)

        self.load_chunks()
        self.initialize_model()
        self.initialize_chromadb()
        self.embed_chunks()
        self.verify_collection()
        self.test_queries()

        print("\n" + "="*70)
        print("✅ EMBEDDING COMPLETE!")
        print(f"📁 Database location: {self.db_path}")
        print(f"📊 Collection name: {self.collection_name}")
        print("="*70 + "\n")


def main():
    """
    Main function for embedding CUBE chunks

    For Google Colab:
    1. Upload cube_optimized_chunks.json to Colab (Files panel on left)
    2. Vector DB will be created in /content/chroma_db (local storage)
    3. Ensure GPU runtime is enabled (Runtime > Change runtime type > T4 GPU)
    4. Download the vector DB folder after completion
    """

    # CONFIGURATION - Update these paths for your environment

    # For Google Colab (uploaded chunks file):
    chunks_file = "/content/cube_optimized_chunks.json"  # After uploading to Colab
    db_path = "/content/chroma_db"  # Vector DB created here

    # For local machine:
    # chunks_file = "/Users/newusername/Desktop/RAG_bankingV.2/Banking-knowledgeAssistance/chunks/cube_optimized_chunks.json"
    # db_path = "/Users/newusername/Desktop/RAG_bankingV.2/Banking-knowledgeAssistance/vector_db/cube_optimized_db"

    collection_name = "cube_docs_optimized"
    model_name = "BAAI/bge-m3"  # Top-tier multilingual embedding model

    # Run embedder
    embedder = CUBEChunkEmbedder(
        chunks_file=chunks_file,
        collection_name=collection_name,
        db_path=db_path,
        model_name=model_name,
        use_gpu=True  # Set to False for CPU-only environments
    )
    embedder.run()

    # To download the vector DB after completion:
    # !zip -r /content/chroma_db.zip /content/chroma_db
    # from google.colab import files
    # files.download('/content/chroma_db.zip')


if __name__ == "__main__":
    main()



CUBE DOCUMENTATION EMBEDDING - BGE-M3 on GPU
📂 Loading chunks from /content/cube_optimized_chunks.json...
✓ Loaded 134 chunks
  - Page chunks: 134
  - Synthetic chunks: 0
  - Avg tokens: 257

🤖 Loading embedding model: BAAI/bge-m3...
   Device: CUDA
   GPU: Tesla T4
   GPU Memory: 14.74 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

✓ Model loaded successfully

🗄️  Initializing ChromaDB at /content/chroma_db...
✓ Created collection: cube_docs_optimized

🔮 Embedding 134 chunks...
   Using GPU-optimized batch size: 16


Preparing chunks: 100%|██████████| 134/134 [00:00<00:00, 82061.14it/s]


Generating embeddings with BGE-M3...


Embedding batches: 100%|██████████| 9/9 [00:14<00:00,  1.63s/it]


Adding to ChromaDB...
✓ Successfully embedded and stored 134 chunks

🔍 Verifying collection...
  Total documents in collection: 134
  Sample document length: 1148 chars
  Sample metadata keys: ['chapter_slug', 'book_slug', 'has_mermaid', 'chapter_id', 'priority', 'book_id', 'page_slug', 'chunk_id', 'account_types', 'shelf_name', 'token_count', 'hierarchy_path', 'chapter_name', 'page_id', 'book_name', 'page_name', 'content_format']

🧪 Testing sample queries...

  Query: 'How to open an NRI account?'
  Top results:
    1. Basic Details Flow (from: Savings Account)
       Preview: DOB• Mobile Number• Email Id• Marital Status• Residencial Status• Account Type• Alternate Number• DC...
    2. Basic Details Flow (from: Delight Savings Account)
       Preview: DOB• Mobile Number• Email Id• Marital Status• Residencial Status• Account Type• Alternate Number• DC...
    3. Basic Details Flow (from: NR Account)
       Preview: DOB• Mobile Number• Email Id• Marital Status• Residencial Status• Accoun

In [ ]:
from google.colab import drive
drive.mount('/content/drive')